# Trial Notebook on Adaptive MH

In [49]:
import os
import gdown
import numpy as np
import torch
from tqdm import tqdm
from scipy.special import logsumexp
import pickle

from podcnf.DataGenerationLinearElasticity import *
from podcnf.NFmodel import NormalizingFlow

In [20]:
if torch.cuda.is_available():
        device = torch.device("cuda")
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
        scaler = torch.amp.GradScaler(enabled=True)
else:
    device = torch.device("cpu")
    scaler = torch.amp.GradScaler(enabled=False)

In [31]:
# {'learning_rate': 0.001, 'num_flows': 16, 'hidden_size': 256, 'hidden_depth': 2, 'weight_decacy': 1e-05}
os.makedirs('../results/elastic', exist_ok=True)
target_folder = os.path.join("..", "results/elastic")
MODEL_NAME = os.path.join(target_folder, 'MODEL_64_NEW.pth')
gdown.download(id = "1Mv9opjkEMDQaLBQx07Fqvh_ItyefLsCl", quiet=True, output = MODEL_NAME)
loaded_model = torch.load(MODEL_NAME, map_location=device)

In [34]:
# Upload scaler for algorithm
with open("../results/elastic/c_scaler.pkl", "rb") as file:
    c_scaler = pickle.load(file)

with open("../results/elastic/mu_scaler.pkl", "rb") as file:
    mu_scaler = pickle.load(file)

In [51]:
dim_x = mu_scaler.n_features_in_
dim_y = c_scaler.n_features_in_

In [52]:
# Linear model
num_flows = 16
hidden_size = 256
hidden_depth = 2

flow = NormalizingFlow(dim_x, dim_y, num_flows, hidden_size, hidden_depth, device).to(device)
flow.load_state_dict(loaded_model)

<All keys matched successfully>

In [11]:
V = torch.load("../results/elastic/V_POD_matrix.pt", weights_only=True)

In [ ]:
Nh = V.shape[0]

1922

In [3]:
num_sensors = 31
sur = [0,1] # top-left already added
for j in range(num_sensors-1):
    xy = np.array([np.pow(j+2,2)+j, np.pow(j+2,2)+j+1])
    sur.extend(xy)
surface_idx = np.array(sur)

In [23]:
P = torch.zeros([Nh,1]).to(device)
P[surface_idx] = int(1)
P[:20]

tensor([[1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.]])

In [24]:
V_s = V[surface_idx, :]
print(V_s.shape)

torch.Size([62, 20])


In [30]:
V.shape, P.shape

(torch.Size([1922, 20]), torch.Size([1922, 1]))

In [14]:
Q = lambda c: c @ V_sensors.T

In [ ]:
def adaptive_metropolis_hastings(
    flow_model, Q, mu_0, u_obs, 
    N, bounds, device,
    temperature=1.0, C_0=None, n_0=100, epsilon=1e-6, s_d=None,
    nrep=100, h=0.1
):

    flow_model.eval()

    # Initialization
    mu_n = mu_0.copy()
    d = len(mu_n)
    mu_n_scaled = mu_scaler.transform(mu_n.reshape(1, -1))

    def compute_log_likelihood(mu_t):
        with torch.no_grad():

            c_samples = flow_model.sample_latent_same_mu(mu_n_scaled, nrep)

            # Projection on the sensors
            # [N_gen, 20] @ [20, 62] -> [N_gen, 62]
            u_sensor_sample = Q(c_samples)

            diff = u_sensor_sample - u_obs.reshape(1, -1)
            dj2 = diff.pow(2).sum(dim=1) # [N_gen]

            # LogSumExp per stabilità (KDE Likelihood)
            # log( sum(exp(-d^2 / 2h^2)) ) - log(N)
            log_pi = torch.logsumexp(-dj2 / (2 * h**2), dim=0) - np.log(n_generations)

            return log_pi.item()

    # Initial value for the Log-likelihood
    log_pi_n = compute_log_likelihood(mu_tensor)

    chain = []
    accepted_count = 0

    # Covariance
    if C_0 is None:
        C_n = np.eye(d) * 1e-5
    else:
        C_n = np.array(C_0)

    mu_bar_n = mu_n.copy()

    # Scaling factor
    if s_d is None:
        scaling_val = (2.38**2) / d
    else:
        scaling_val = s_d

    # MCMC LOOP
    for n in tqdm(range(1, N + 1), desc="Adaptive MH"):

        # Proposal Covariance once n>n_0
        if n <= n_0:
            proposal_cov = C_n
        else:
            proposal_cov = scaling_val * C_n + epsilon * np.eye(d)

        perturbation = np.random.multivariate_normal(np.zeros(d), proposal_cov)
        Y = mu_n + perturbation

        # Check Prior (Bounds)
        if (Y[0] < bounds['m_min'] or Y[0] > bounds['m_max'] or
            Y[1] < bounds['d_min'] or Y[1] > bounds['d_max']):
            chain.append(mu_n)
            mu_next = mu_n
        else:
            # Compute the likelihood for the candidate
            Y_scaled = mu_scaler.transform(Y.reshape(1, -1))
            Y_tensor = torch.tensor(Y_scaled, dtype=torch.float32).to(device)

            log_pi_Y = compute_log_likelihood(Y_tensor)

            # Acceptance ratio
            log_alpha = (log_pi_Y - log_pi_n) / temperature

            if np.log(np.random.rand()) < log_alpha:
                mu_n = Y
                log_pi_n = log_pi_Y
                accepted_count += 1

            chain.append(mu_n)
            mu_next = mu_n

        # Updating the covariance
        if n > n_0:
            mu_bar_prev = mu_bar_n.copy()
            mu_bar_n = (n * mu_bar_prev + mu_next) / (n + 1)

            dt = (mu_next - mu_bar_prev).reshape(-1, 1)
            term_update = np.dot(dt, dt.T)
            C_n = ((n - 1) / n) * C_n + (scaling_val / n) * (term_update * (n / (n + 1)) + epsilon * np.eye(d))

    chain = np.array(chain)
    acc_rate = accepted_count / N
    print(f"\nAcceptance Rate: {acc_rate:.2%}")
    return chain, C_n